# BERT E2E ABSA - Kaggle Training Notebook
Aspect-Based Sentiment Analysis sử dụng BERT fine-tuning.

> Notebook này tự chứa toàn bộ source code, chạy trực tiếp trong cells — không cần upload file `.py` riêng.
>
> **Tiếng Việt:** đổi `MODEL_NAME = 'vinai/phobert-base'` và `DO_LOWER_CASE = False`

## 1. Cài đặt dependencies

In [1]:
!pip install transformers tensorboardX tqdm pytorch-crf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.2 MB/s eta 0:00:00


## 2. Kiểm tra GPU

In [2]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2.1. Convert CSV to text

In [3]:
CSV_PATH = ''
TEXT_PATH = ''

In [4]:
import pandas as pd

def csv_to_txt(csv_path, txt_path):
    df = pd.read_csv(csv_path)
    df = df[df['polarity'] != 'conflict']
    
    with open(txt_path, 'w', encoding='utf-8') as f:
        for _, group in df.groupby('Sentence'):
            sentence = group.iloc[0]['Sentence']
            words = sentence.split()
            tags = ['O'] * len(words)
            
            for _, row in group.iterrows():
                aspect_words = row['Aspect Term'].split()
                # tìm vị trí aspect trong câu
                for i in range(len(words) - len(aspect_words) + 1):
                    if words[i:i+len(aspect_words)] == aspect_words:
                        pol = row['polarity'][0:3].upper()  # positive→POS
                        for j, idx in enumerate(range(i, i+len(aspect_words))):
                            tags[idx] = 'T-' + pol  # OT format
                        break
            
            tag_string = ' '.join(f'{w}={t}' for w, t in zip(words, tags))
            f.write(f'{sentence}####{tag_string}\n')


## 3. Sequence Utility Functions
Các hàm chuyển đổi giữa các tagging schema (OT, BIO, BIEOS) và trích xuất targeted sentiment.

In [5]:
import numpy as np


def ot2bieos_ts(ts_tag_sequence):
    """Convert OT tag sequence to BIEOS format."""
    n_tags = len(ts_tag_sequence)
    new_ts_sequence = []
    prev_pos = '$$$'
    for i in range(n_tags):
        cur_ts_tag = ts_tag_sequence[i]
        if cur_ts_tag == 'O' or cur_ts_tag == 'EQ':
            new_ts_sequence.append('O')
            cur_pos = 'O'
        else:
            cur_pos, cur_sentiment = cur_ts_tag.split('-')
            if cur_pos != prev_pos:
                if i == n_tags - 1:
                    new_ts_sequence.append('S-%s' % cur_sentiment)
                else:
                    next_ts_tag = ts_tag_sequence[i + 1]
                    if next_ts_tag == 'O':
                        new_ts_sequence.append('S-%s' % cur_sentiment)
                    else:
                        new_ts_sequence.append('B-%s' % cur_sentiment)
            else:
                if i == n_tags - 1:
                    new_ts_sequence.append('E-%s' % cur_sentiment)
                else:
                    next_ts_tag = ts_tag_sequence[i + 1]
                    if next_ts_tag == 'O':
                        new_ts_sequence.append('E-%s' % cur_sentiment)
                    else:
                        new_ts_sequence.append('I-%s' % cur_sentiment)
        prev_pos = cur_pos
    return new_ts_sequence


def ot2bio_ts(ts_tag_sequence):
    """Convert OT tag sequence to BIO format."""
    new_ts_sequence = []
    n_tag = len(ts_tag_sequence)
    prev_pos = '$$$'
    for i in range(n_tag):
        cur_ts_tag = ts_tag_sequence[i]
        if cur_ts_tag == 'O':
            new_ts_sequence.append('O')
            cur_pos = 'O'
        else:
            cur_pos, cur_sentiment = cur_ts_tag.split('-')
            if cur_pos == prev_pos:
                new_ts_sequence.append('I-%s' % cur_sentiment)
            else:
                new_ts_sequence.append('B-%s' % cur_sentiment)
        prev_pos = cur_pos
    return new_ts_sequence


def bio2ot_ts(ts_tag_sequence):
    """Convert BIO tag sequence back to OT format."""
    new_ts_sequence = []
    n_tags = len(ts_tag_sequence)
    for i in range(n_tags):
        ts_tag = ts_tag_sequence[i]
        if ts_tag == 'O' or ts_tag == 'EQ':
            new_ts_sequence.append('O')
        else:
            pos, sentiment = ts_tag.split('-')
            new_ts_sequence.append('T-%s' % sentiment)
    return new_ts_sequence


def tag2ts(ts_tag_sequence):
    """Convert BIEOS tag sequence to list of (begin, end, sentiment)."""
    n_tags = len(ts_tag_sequence)
    ts_sequence, sentiments = [], []
    beg, end = -1, -1
    for i in range(n_tags):
        ts_tag = ts_tag_sequence[i]
        eles = ts_tag.split('-')
        if len(eles) == 2:
            pos, sentiment = eles
        else:
            pos, sentiment = 'O', 'O'
        if sentiment != 'O':
            sentiments.append(sentiment)
        if pos == 'S':
            ts_sequence.append((i, i, sentiment))
            sentiments = []
        elif pos == 'B':
            beg = i
            if len(sentiments) > 1:
                sentiments = [sentiments[-1]]
        elif pos == 'E':
            end = i
            if end > beg > -1 and len(set(sentiments)) == 1:
                ts_sequence.append((beg, end, sentiment))
                sentiments = []
                beg, end = -1, -1
    return ts_sequence


print('✓ Sequence utility functions loaded.')

✓ Sequence utility functions loaded.


## 4. Data Processing
Classes và hàm xử lý dữ liệu ABSA: đọc data, tokenize, tạo features, đánh giá kết quả.

In [6]:
import csv
import logging
import os
import sys
from io import open

logger = logging.getLogger(__name__)
SMALL_POSITIVE_CONST = 1e-4


class InputExample(object):
    """A single training/test example for simple sequence classification."""
    def __init__(self, guid, text_a, text_b=None, label=None):
        self.guid = guid
        self.text_a = text_a
        self.text_b = text_b
        self.label = label


class SeqInputFeatures(object):
    """A single set of features of data for the ABSA task."""
    def __init__(self, input_ids, input_mask, segment_ids, label_ids, evaluate_label_ids):
        self.input_ids = input_ids
        self.input_mask = input_mask
        self.segment_ids = segment_ids
        self.label_ids = label_ids
        self.evaluate_label_ids = evaluate_label_ids


class DataProcessor(object):
    """Base class for data converters."""
    def get_train_examples(self, data_dir): raise NotImplementedError()
    def get_dev_examples(self, data_dir): raise NotImplementedError()
    def get_test_examples(self, data_dir): raise NotImplementedError()
    def get_labels(self): raise NotImplementedError()


class ABSAProcessor(DataProcessor):
    """Processor for the ABSA datasets."""
    def get_train_examples(self, data_dir, tagging_schema):
        return self._create_examples(data_dir, 'train', tagging_schema)
    def get_dev_examples(self, data_dir, tagging_schema):
        return self._create_examples(data_dir, 'dev', tagging_schema)
    def get_test_examples(self, data_dir, tagging_schema):
        return self._create_examples(data_dir, 'test', tagging_schema)

    def get_labels(self, tagging_schema):
        if tagging_schema == 'OT': return []
        elif tagging_schema == 'BIO':
            return ['O', 'EQ', 'B-POS', 'I-POS', 'B-NEG', 'I-NEG', 'B-NEU', 'I-NEU']
        elif tagging_schema == 'BIEOS':
            return ['O', 'EQ', 'B-POS', 'I-POS', 'E-POS', 'S-POS',
                    'B-NEG', 'I-NEG', 'E-NEG', 'S-NEG',
                    'B-NEU', 'I-NEU', 'E-NEU', 'S-NEU']
        else: raise Exception('Invalid tagging schema %s...' % tagging_schema)

    def _create_examples(self, data_dir, set_type, tagging_schema):
        examples = []
        file = os.path.join(data_dir, '%s.txt' % set_type)
        class_count = np.zeros(3)
        with open(file, 'r', encoding='UTF-8') as fp:
            sample_id = 0
            for line in fp:
                sent_string, tag_string = line.strip().split('####')
                words, tags = [], []
                for tag_item in tag_string.split(' '):
                    eles = tag_item.split('=')
                    if len(eles) == 1: raise Exception('Invalid samples %s...' % tag_string)
                    elif len(eles) == 2: word, tag = eles
                    else: word, tag = ''.join((len(eles)-2)*['=']), eles[-1]
                    words.append(word); tags.append(tag)
                if tagging_schema == 'BIEOS': tags = ot2bieos_ts(tags)
                elif tagging_schema == 'BIO': tags = ot2bio_ts(tags)
                guid = '%s-%s' % (set_type, sample_id)
                text_a = ' '.join(words)
                gold_ts = tag2ts(ts_tag_sequence=tags)
                for (b, e, s) in gold_ts:
                    if s == 'POS': class_count[0] += 1
                    if s == 'NEG': class_count[1] += 1
                    if s == 'NEU': class_count[2] += 1
                examples.append(InputExample(guid=guid, text_a=text_a, text_b=None, label=tags))
                sample_id += 1
        print('%s class count: %s' % (set_type, class_count))
        return examples


processors = {
    'laptop14': ABSAProcessor, 'rest_total': ABSAProcessor,
    'rest_total_revised': ABSAProcessor, 'rest14': ABSAProcessor,
    'rest15': ABSAProcessor, 'rest16': ABSAProcessor, 'rest14-ano': ABSAProcessor
}

print('✓ Data processing classes loaded.')

✓ Data processing classes loaded.


In [7]:
def convert_examples_to_seq_features(examples, label_list, tokenizer,
                                     cls_token_at_end=False, pad_on_left=False, cls_token='[CLS]',
                                     sep_token='[SEP]', pad_token=0, sequence_a_segment_id=0,
                                     sequence_b_segment_id=1, cls_token_segment_id=1, pad_token_segment_id=0,
                                     mask_padding_with_zero=True):
    """Feature extraction for sequence labeling."""
    label_map = {label: i for i, label in enumerate(label_list)}
    features, examples_tokenized = [], []
    max_seq_length = -1
    for (ex_index, example) in enumerate(examples):
        tokens_a, labels_a, evaluate_label_ids = [], [], []
        words = example.text_a.split(' ')
        wid, tid = 0, 0
        for word, label in zip(words, example.label):
            subwords = tokenizer.tokenize(word)
            tokens_a.extend(subwords)
            if label != 'O': labels_a.extend([label] + ['EQ'] * (len(subwords) - 1))
            else: labels_a.extend(['O'] * len(subwords))
            evaluate_label_ids.append(tid)
            wid += 1; tid += len(subwords)
        assert tid == len(tokens_a)
        evaluate_label_ids = np.array(evaluate_label_ids, dtype=np.int32)
        examples_tokenized.append((tokens_a, labels_a, evaluate_label_ids))
        if len(tokens_a) > max_seq_length: max_seq_length = len(tokens_a)
    max_seq_length += 2  # [CLS] and [SEP]
    for ex_index, (tokens_a, labels_a, evaluate_label_ids) in enumerate(examples_tokenized):
        tokens = tokens_a + [sep_token]
        segment_ids = [sequence_a_segment_id] * len(tokens)
        labels = labels_a + ['O']
        if cls_token_at_end:
            tokens = tokens + [cls_token]; segment_ids = segment_ids + [cls_token_segment_id]; labels = labels + ['O']
        else:
            tokens = [cls_token] + tokens; segment_ids = [cls_token_segment_id] + segment_ids
            labels = ['O'] + labels; evaluate_label_ids += 1
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        input_mask = [1 if mask_padding_with_zero else 0] * len(input_ids)
        padding_length = max_seq_length - len(input_ids)
        label_ids = [label_map[label] for label in labels]
        if pad_on_left:
            input_ids = ([pad_token]*padding_length) + input_ids
            input_mask = ([0 if mask_padding_with_zero else 1]*padding_length) + input_mask
            segment_ids = ([pad_token_segment_id]*padding_length) + segment_ids
            label_ids = ([0]*padding_length) + label_ids; evaluate_label_ids += padding_length
        else:
            input_ids = input_ids + ([pad_token]*padding_length)
            input_mask = input_mask + ([0 if mask_padding_with_zero else 1]*padding_length)
            segment_ids = segment_ids + ([pad_token_segment_id]*padding_length)
            label_ids = label_ids + ([0]*padding_length)
        assert len(input_ids) == max_seq_length
        assert len(input_mask) == max_seq_length
        assert len(segment_ids) == max_seq_length
        assert len(label_ids) == max_seq_length
        features.append(SeqInputFeatures(input_ids=input_ids, input_mask=input_mask,
                         segment_ids=segment_ids, label_ids=label_ids, evaluate_label_ids=evaluate_label_ids))
    print('maximal sequence length is', max_seq_length)
    return features


def match_ts(gold_ts_sequence, pred_ts_sequence):
    """Calculate the number of correctly predicted targeted sentiment."""
    tag2tagid = {'POS': 0, 'NEG': 1, 'NEU': 2}
    hit_count, gold_count, pred_count = np.zeros(3), np.zeros(3), np.zeros(3)
    for t in gold_ts_sequence:
        tid = tag2tagid[t[2]]; gold_count[tid] += 1
    for t in pred_ts_sequence:
        tid = tag2tagid[t[2]]
        if t in gold_ts_sequence: hit_count[tid] += 1
        pred_count[tid] += 1
    return hit_count, gold_count, pred_count


def compute_metrics_absa(preds, labels, all_evaluate_label_ids, tagging_schema):
    """Compute micro/macro F1 for ABSA task."""
    if tagging_schema == 'BIEOS':
        absa_label_vocab = {'O':0,'EQ':1,'B-POS':2,'I-POS':3,'E-POS':4,'S-POS':5,
                            'B-NEG':6,'I-NEG':7,'E-NEG':8,'S-NEG':9,
                            'B-NEU':10,'I-NEU':11,'E-NEU':12,'S-NEU':13}
    elif tagging_schema == 'BIO':
        absa_label_vocab = {'O':0,'EQ':1,'B-POS':2,'I-POS':3,'B-NEG':4,'I-NEG':5,'B-NEU':6,'I-NEU':7}
    elif tagging_schema == 'OT':
        absa_label_vocab = {'O':0,'EQ':1,'T-POS':2,'T-NEG':3,'T-NEU':4}
    else: raise Exception('Invalid tagging schema %s...' % tagging_schema)
    absa_id2tag = {v: k for k, v in absa_label_vocab.items()}
    n_tp_ts, n_gold_ts, n_pred_ts = np.zeros(3), np.zeros(3), np.zeros(3)
    ts_precision, ts_recall, ts_f1 = np.zeros(3), np.zeros(3), np.zeros(3)
    n_samples = len(all_evaluate_label_ids)
    class_count = np.zeros(3)
    for i in range(n_samples):
        evaluate_label_ids = all_evaluate_label_ids[i]
        pred_labels = preds[i][evaluate_label_ids]
        gold_labels = labels[i][evaluate_label_ids]
        assert len(pred_labels) == len(gold_labels)
        pred_tags = [absa_id2tag[label] for label in pred_labels]
        gold_tags = [absa_id2tag[label] for label in gold_labels]
        if tagging_schema == 'OT':
            gold_tags = ot2bieos_ts(gold_tags); pred_tags = ot2bieos_ts(pred_tags)
        elif tagging_schema == 'BIO':
            gold_tags = ot2bieos_ts(bio2ot_ts(gold_tags)); pred_tags = ot2bieos_ts(bio2ot_ts(pred_tags))
        g_ts = tag2ts(ts_tag_sequence=gold_tags); p_ts = tag2ts(ts_tag_sequence=pred_tags)
        hit, gold, pred = match_ts(gold_ts_sequence=g_ts, pred_ts_sequence=p_ts)
        n_tp_ts += hit; n_gold_ts += gold; n_pred_ts += pred
        for (b, e, s) in g_ts:
            if s == 'POS': class_count[0] += 1
            if s == 'NEG': class_count[1] += 1
            if s == 'NEU': class_count[2] += 1
    for i in range(3):
        ts_precision[i] = float(n_tp_ts[i]) / float(n_pred_ts[i] + SMALL_POSITIVE_CONST)
        ts_recall[i] = float(n_tp_ts[i]) / float(n_gold_ts[i] + SMALL_POSITIVE_CONST)
        ts_f1[i] = 2*ts_precision[i]*ts_recall[i] / (ts_precision[i]+ts_recall[i]+SMALL_POSITIVE_CONST)
    macro_f1 = ts_f1.mean()
    n_tp_total = sum(n_tp_ts); n_g_total = sum(n_gold_ts); n_p_total = sum(n_pred_ts)
    print('class_count:', class_count)
    micro_p = float(n_tp_total) / (n_p_total + SMALL_POSITIVE_CONST)
    micro_r = float(n_tp_total) / (n_g_total + SMALL_POSITIVE_CONST)
    micro_f1 = 2*micro_p*micro_r / (micro_p+micro_r+SMALL_POSITIVE_CONST)
    return {'macro-f1': macro_f1, 'precision': micro_p, 'recall': micro_r, 'micro-f1': micro_f1}


print('✓ Feature conversion & metrics functions loaded.')

✓ Feature conversion & metrics functions loaded.


## 5. Model Architecture
Các lớp mô hình ABSA: TaggerConfig, SAN, GRU, LSTM, BertABSATagger.

In [8]:
import os
import sys
import torch
import torch.nn as nn
from transformers import BertPreTrainedModel, BertModel, BertConfig
from torch.nn import CrossEntropyLoss
from torchcrf import CRF as TorchCRF

# Fix: Jupyter notebook __main__ không có __file__,
# nhưng transformers mới cần đọc nó trong post_init()
# Thay vì gán một file ảo 'notebook', ta tạo 1 file rỗng thật sự để nó đọc
if not hasattr(sys.modules['__main__'], '__file__'):
    dummy_file = os.path.abspath('dummy_transformers_fix.py')
    with open(dummy_file, 'w') as f: f.write('')
    sys.modules['__main__'].__file__ = dummy_file

print('✓ Imports ready (with Jupyter __file__ fix).')

✓ Imports ready (with Jupyter __file__ fix).


In [9]:
class TaggerConfig:
    def __init__(self):
        self.hidden_dropout_prob = 0.1
        self.hidden_size = 768
        self.n_rnn_layers = 1
        self.bidirectional = True


class SAN(nn.Module):
    """Self Attention Network."""
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        self.d_model = d_model; self.nhead = nhead
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.dropout = nn.Dropout(p=dropout); self.norm = nn.LayerNorm(d_model)
    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        src2, _ = self.self_attn(src, src, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)
        src = src + self.dropout(src2)
        return self.norm(src)


class GRU(nn.Module):
    """Customized GRU with layer normalization."""
    def __init__(self, input_size, hidden_size, bidirectional=True):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size // 2 if bidirectional else hidden_size
        self.bidirectional = bidirectional
        self.Wxrz = nn.Linear(self.input_size, 2*self.hidden_size, bias=True)
        self.Whrz = nn.Linear(self.hidden_size, 2*self.hidden_size, bias=True)
        self.Wxn = nn.Linear(self.input_size, self.hidden_size, bias=True)
        self.Whn = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.LNx1 = nn.LayerNorm(2*self.hidden_size); self.LNh1 = nn.LayerNorm(2*self.hidden_size)
        self.LNx2 = nn.LayerNorm(self.hidden_size); self.LNh2 = nn.LayerNorm(self.hidden_size)

    def forward(self, x):
        def recurrence(xt, htm1):
            gates_rz = torch.sigmoid(self.LNx1(self.Wxrz(xt)) + self.LNh1(self.Whrz(htm1)))
            rt, zt = gates_rz.chunk(2, 1)
            nt = torch.tanh(self.LNx2(self.Wxn(xt)) + rt * self.LNh2(self.Whn(htm1)))
            return (1.0 - zt) * nt + zt * htm1
        steps = range(x.size(1)); bs = x.size(0)
        hidden = torch.zeros(bs, self.hidden_size).cuda()
        inp = x.transpose(0, 1); output = []
        for t in steps: hidden = recurrence(inp[t], hidden); output.append(hidden)
        output = torch.stack(output, 0).transpose(0, 1)
        if self.bidirectional:
            output_b = []; hidden_b = torch.zeros(bs, self.hidden_size).cuda()
            for t in steps[::-1]: hidden_b = recurrence(inp[t], hidden_b); output_b.append(hidden_b)
            output_b = torch.stack(output_b[::-1], 0).transpose(0, 1)
            output = torch.cat([output, output_b], dim=-1)
        return output, None


class LSTM(nn.Module):
    """Customized LSTM with layer normalization."""
    def __init__(self, input_size, hidden_size, bidirectional=True):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size // 2 if bidirectional else hidden_size
        self.bidirectional = bidirectional
        self.LNx = nn.LayerNorm(4*self.hidden_size); self.LNh = nn.LayerNorm(4*self.hidden_size)
        self.LNc = nn.LayerNorm(self.hidden_size)
        self.Wx = nn.Linear(self.input_size, 4*self.hidden_size, bias=True)
        self.Wh = nn.Linear(self.hidden_size, 4*self.hidden_size, bias=True)

    def forward(self, x):
        def recurrence(xt, hidden):
            htm1, ctm1 = hidden
            gates = self.LNx(self.Wx(xt)) + self.LNh(self.Wh(htm1))
            it, ft, gt, ot = gates.chunk(4, 1)
            it = torch.sigmoid(it); ft = torch.sigmoid(ft)
            gt = torch.tanh(gt); ot = torch.sigmoid(ot)
            ct = (ft * ctm1) + (it * gt)
            ht = ot * torch.tanh(self.LNc(ct))
            return ht, ct
        steps = range(x.size(1))
        h0 = torch.zeros(x.size(0), self.hidden_size).cuda()
        c0 = torch.zeros(x.size(0), self.hidden_size).cuda()
        hidden = (h0, c0); inp = x.transpose(0, 1); output = []
        for t in steps: hidden = recurrence(inp[t], hidden); output.append(hidden[0])
        output = torch.stack(output, 0).transpose(0, 1)
        if self.bidirectional:
            hidden_b = (torch.zeros_like(h0), torch.zeros_like(c0)); output_b = []
            for t in steps[::-1]: hidden_b = recurrence(inp[t], hidden_b); output_b.append(hidden_b[0])
            output_b = torch.stack(output_b[::-1], 0).transpose(0, 1)
            output = torch.cat([output, output_b], dim=-1)
        return output, None


print('✓ Tagger components loaded (TaggerConfig, SAN, GRU, LSTM).')

✓ Tagger components loaded (TaggerConfig, SAN, GRU, LSTM).


In [10]:
class BertABSATagger(BertPreTrainedModel):
    """BERT-based Aspect-Based Sentiment Analysis Tagger."""
    _tied_weights_keys = []

    @property
    def all_tied_weights_keys(self):
        return {}

    def mark_tied_weights_as_initialized(self):
        pass

    def __init__(self, bert_config):
        super().__init__(bert_config)
        self.num_labels = bert_config.num_labels
        self.tagger_config = TaggerConfig()
        self.tagger_config.absa_type = bert_config.absa_type.lower()
        if bert_config.tfm_mode == 'finetune': self.bert = BertModel(bert_config)
        else: raise Exception('Invalid transformer mode %s!!!' % bert_config.tfm_mode)
        self.bert_dropout = nn.Dropout(bert_config.hidden_dropout_prob)
        if bert_config.fix_tfm:
            for p in self.bert.parameters(): p.requires_grad = False
        self.tagger = None
        if self.tagger_config.absa_type == 'linear':
            penultimate_hidden_size = bert_config.hidden_size
        else:
            self.tagger_dropout = nn.Dropout(self.tagger_config.hidden_dropout_prob)
            if self.tagger_config.absa_type == 'lstm':
                self.tagger = LSTM(bert_config.hidden_size, self.tagger_config.hidden_size, self.tagger_config.bidirectional)
            elif self.tagger_config.absa_type == 'gru':
                self.tagger = GRU(bert_config.hidden_size, self.tagger_config.hidden_size, self.tagger_config.bidirectional)
            elif self.tagger_config.absa_type == 'tfm':
                self.tagger = nn.TransformerEncoderLayer(d_model=bert_config.hidden_size, nhead=12,
                                                         dim_feedforward=4*bert_config.hidden_size, dropout=0.1)
            elif self.tagger_config.absa_type == 'san':
                self.tagger = SAN(d_model=bert_config.hidden_size, nhead=12, dropout=0.1)
            elif self.tagger_config.absa_type == 'crf':
                self.tagger = TorchCRF(self.num_labels, batch_first=True)
            else: raise Exception('Unimplemented downstream tagger %s...' % self.tagger_config.absa_type)
            penultimate_hidden_size = self.tagger_config.hidden_size
        self.classifier = nn.Linear(penultimate_hidden_size, bert_config.num_labels)

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None,
                position_ids=None, head_mask=None):
        outputs = self.bert(input_ids, position_ids=position_ids, token_type_ids=token_type_ids,
                            attention_mask=attention_mask, head_mask=head_mask)
        tagger_input = self.bert_dropout(outputs[0])
        if self.tagger is None or self.tagger_config.absa_type == 'crf':
            logits = self.classifier(tagger_input)
        else:
            if self.tagger_config.absa_type in ('lstm', 'gru'):
                classifier_input, _ = self.tagger(tagger_input)
            elif self.tagger_config.absa_type in ('san', 'tfm'):
                tagger_input = tagger_input.transpose(0, 1)
                classifier_input = self.tagger(tagger_input).transpose(0, 1)
            else: raise Exception('Unimplemented downstream tagger %s...' % self.tagger_config.absa_type)
            classifier_input = self.tagger_dropout(classifier_input)
            logits = self.classifier(classifier_input)
        outputs = (logits,) + outputs[2:]
        if labels is not None:
            if self.tagger_config.absa_type != 'crf':
                loss_fct = CrossEntropyLoss()
                if attention_mask is not None:
                    active_loss = attention_mask.view(-1) == 1
                    active_logits = logits.view(-1, self.num_labels)[active_loss]
                    active_labels = labels.view(-1)[active_loss]
                    loss = loss_fct(active_logits, active_labels)
                else: loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            else: loss = -self.tagger(logits, labels, mask=attention_mask.bool())
            outputs = (loss,) + outputs
        return outputs

    def viterbi_tags(self, logits, mask):
        return self.tagger.decode(logits, mask=mask.bool())


print('✓ BertABSATagger loaded.')

✓ BertABSATagger loaded.


## 6. Cấu hình
Chỉnh các tham số tại đây trước khi chạy.

In [11]:
import os

# Auto-detect môi trường
ON_KAGGLE = os.path.exists('/kaggle/working')
KAGGLE_INPUT = '/kaggle/input'
WORKING_DIR  = '/kaggle/working' if ON_KAGGLE else os.getcwd()
print('Environment:', 'Kaggle' if ON_KAGGLE else 'Local')

# ============================================================
# CẤU HÌNH CHÍNH - chỉnh tại đây
# ============================================================
TASK_NAME        = 'rest14-ano'   
MODEL_NAME       = 'bert-base-uncased'
ABSA_TYPE        = 'linear'   # linear | lstm | gru | san | tfm | crf
TAGGING_SCHEMA   = 'BIEOS'
MAX_STEPS        = 1500
SAVE_STEPS       = 500
LOG_STEPS        = 100
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE  = 8
LEARNING_RATE    = 2e-5
SEED             = 42
DO_LOWER_CASE    = True

# Path data — đổi dataset slug nếu cần
if ON_KAGGLE:
    # Tên dataset slug trên Kaggle: Settings > Add Data
    DATA_DIR   = f'{KAGGLE_INPUT}/datasets/conanwinner1/{TASK_NAME}'
    OUTPUT_DIR = f'{WORKING_DIR}/bert-{ABSA_TYPE}-{TASK_NAME}'
    CACHE_DIR  = f'{WORKING_DIR}/hf-cache'
else:
    DATA_DIR   = os.path.join(os.path.dirname(os.getcwd()), 'Final', 'data', TASK_NAME)
    OUTPUT_DIR = os.path.join(WORKING_DIR, f'bert-{ABSA_TYPE}-{TASK_NAME}')
    CACHE_DIR  = os.path.join(WORKING_DIR, 'hf-cache')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)
# ============================================================

print(f'  Task      : {TASK_NAME}')
print(f'  Model     : {MODEL_NAME}')
print(f'  Tagger    : {ABSA_TYPE}')
print(f'  Schema    : {TAGGING_SCHEMA}')
print(f'  Max steps : {MAX_STEPS}')
print(f'  Data dir  : {DATA_DIR}')
print(f'  Output dir: {OUTPUT_DIR}')
print(f'  Cache dir : {CACHE_DIR}')


Environment: Kaggle
  Task      : rest14-ano
  Model     : bert-base-uncased
  Tagger    : linear
  Schema    : BIEOS
  Max steps : 1500
  Data dir  : /kaggle/input/datasets/conanwinner1/semeval14-absa/rest14-ano
  Output dir: /kaggle/working/bert-linear-rest14-ano
  Cache dir : /kaggle/working/hf-cache


## 7. Import và khởi tạo

In [12]:
import random
import logging
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
from transformers import BertConfig, BertTokenizer, get_linear_schedule_with_warmup, WEIGHTS_NAME
from torch.optim import AdamW
from tensorboardX import SummaryWriter
from tqdm.notebook import tqdm, trange

logging.basicConfig(format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count()
print(f'Using device: {device}, n_gpu: {n_gpu}')

Using device: cuda, n_gpu: 2


## 8. Khởi tạo model và tokenizer

In [13]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)
processor  = processors[TASK_NAME]()
label_list = processor.get_labels(TAGGING_SCHEMA)
num_labels = len(label_list)
print(f'Labels ({num_labels}): {label_list}')

# Load BertConfig từ pretrained, gắn thêm custom attrs
config = BertConfig.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    finetuning_task=TASK_NAME, cache_dir=CACHE_DIR
)
config.absa_type = ABSA_TYPE
config.tfm_mode  = 'finetune'
config.fix_tfm   = 0

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME, do_lower_case=DO_LOWER_CASE, cache_dir=CACHE_DIR
)

# BertABSATagger là custom class — không thể dùng from_pretrained(MODEL_NAME) trực tiếp
# vì HuggingFace sẽ tìm config.architectures và không tìm thấy BertABSATagger.
# Cách đúng: khởi tạo model với config, rồi load BERT weights vào phần bert sub-module.
model = BertABSATagger(config)
bert_state = BertModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR).state_dict()
missing, unexpected = model.bert.load_state_dict(bert_state, strict=False)
if missing:    print(f'Missing keys  : {missing[:5]}...')
if unexpected: print(f'Unexpected keys: {unexpected[:5]}...')

model.to(device)
if n_gpu > 1: model = torch.nn.DataParallel(model)
print('Model loaded successfully.')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


Labels (14): ['O', 'EQ', 'B-POS', 'I-POS', 'E-POS', 'S-POS', 'B-NEG', 'I-NEG', 'E-NEG', 'S-NEG', 'B-NEU', 'I-NEU', 'E-NEU', 'S-NEU']


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.
Trainable params: 109,493,006


## 9. Load và cache dữ liệu

In [14]:
# DATA_DIR đã được set tự động ở cell Config (cell 6) dựa theo môi trường.
# Nếu chạy trên Kaggle: vào Settings > Add Data, thêm dataset chứa thư mục laptop14/
# rồi cập nhật slug trong biến DATA_DIR ở cell Config nếu cần.
print(f'Using DATA_DIR: {DATA_DIR}')
assert os.path.isdir(DATA_DIR), f'DATA_DIR không tồn tại: {DATA_DIR}'


Using DATA_DIR: /kaggle/input/datasets/conanwinner1/semeval14-absa/rest14-ano


AssertionError: DATA_DIR không tồn tại: /kaggle/input/datasets/conanwinner1/semeval14-absa/rest14-ano

In [ ]:
def load_dataset(data_dir, task, tokenizer, mode='train'):
    proc   = processors[task]()
    labels = proc.get_labels(TAGGING_SCHEMA)
    if mode == 'train':   examples = proc.get_train_examples(data_dir, TAGGING_SCHEMA)
    elif mode == 'dev':   examples = proc.get_dev_examples(data_dir, TAGGING_SCHEMA)
    else:                 examples = proc.get_test_examples(data_dir, TAGGING_SCHEMA)

    features = convert_examples_to_seq_features(
        examples=examples, label_list=labels, tokenizer=tokenizer,
        cls_token=tokenizer.cls_token, sep_token=tokenizer.sep_token,
        cls_token_segment_id=0, pad_on_left=False, pad_token_segment_id=0)

    all_input_ids          = torch.tensor([f.input_ids   for f in features], dtype=torch.long)
    all_input_mask         = torch.tensor([f.input_mask  for f in features], dtype=torch.long)
    all_segment_ids        = torch.tensor([f.segment_ids for f in features], dtype=torch.long)
    all_label_ids          = torch.tensor([f.label_ids   for f in features], dtype=torch.long)
    all_evaluate_label_ids = [f.evaluate_label_ids for f in features]
    return TensorDataset(all_input_ids, all_input_mask, all_segment_ids, all_label_ids), all_evaluate_label_ids

train_dataset, _ = load_dataset(DATA_DIR, TASK_NAME, tokenizer, mode='train')
print(f'Train samples: {len(train_dataset)}')


## 10. Training

In [ ]:
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=TRAIN_BATCH_SIZE)
t_total = MAX_STEPS
num_train_epochs = MAX_STEPS // len(train_dataloader) + 1

no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=t_total)
print(f'Training for {t_total} steps (~{num_train_epochs} epochs)')

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
tb_writer = SummaryWriter(log_dir=OUTPUT_DIR)
global_step = 0
tr_loss, logging_loss = 0.0, 0.0
model.zero_grad(); set_seed(SEED)

for epoch in trange(num_train_epochs, desc='Epoch'):
    for step, batch in enumerate(tqdm(train_dataloader, desc='Iteration')):
        model.train()
        batch = tuple(t.to(device) for t in batch)
        inputs = {'input_ids': batch[0], 'attention_mask': batch[1], 'token_type_ids': batch[2], 'labels': batch[3]}
        outputs = model(**inputs)
        loss = outputs[0]
        if n_gpu > 1: loss = loss.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        tr_loss += loss.item()
        optimizer.step(); scheduler.step(); model.zero_grad(); global_step += 1
        if global_step % LOG_STEPS == 0:
            avg_loss = (tr_loss - logging_loss) / LOG_STEPS
            tb_writer.add_scalar('loss', avg_loss, global_step)
            tb_writer.add_scalar('lr', scheduler.get_last_lr()[0], global_step)
            print(f'  Step {global_step} | loss: {avg_loss:.4f} | lr: {scheduler.get_last_lr()[0]:.2e}')
            logging_loss = tr_loss
        if global_step % SAVE_STEPS == 0:
            ckpt_dir = os.path.join(OUTPUT_DIR, f'checkpoint-{global_step}')
            os.makedirs(ckpt_dir, exist_ok=True)
            m = model.module if hasattr(model, 'module') else model
            m.save_pretrained(ckpt_dir)
            print(f'  Checkpoint saved: {ckpt_dir}')
        if global_step >= MAX_STEPS: break
    if global_step >= MAX_STEPS: break

In [ ]:
# Lưu model cuối
m = model.module if hasattr(model, 'module') else model
m.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\nTraining done. Model saved to: {OUTPUT_DIR}')
tb_writer.close()

## 11. Evaluation (Dev & Test)

In [ ]:
def evaluate(model, tokenizer, data_dir, task, mode='test'):
    dataset, evaluate_label_ids = load_dataset(data_dir, task, tokenizer, mode=mode)
    dataloader = DataLoader(dataset, sampler=SequentialSampler(dataset), batch_size=EVAL_BATCH_SIZE)
    eval_loss = 0.0; preds = None; out_label_ids = None
    crf_logits, crf_mask = [], []
    model.eval()
    for batch in tqdm(dataloader, desc=f'Evaluating [{mode}]'):
        batch = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            inputs = {'input_ids': batch[0], 'attention_mask': batch[1], 'token_type_ids': batch[2], 'labels': batch[3]}
            outputs = model(**inputs)
            tmp_loss, logits = outputs[:2]
            eval_loss += tmp_loss.mean().item()
            crf_logits.append(logits); crf_mask.append(batch[1])
        if preds is None:
            preds = logits.detach().cpu().numpy()
            out_label_ids = inputs['labels'].detach().cpu().numpy()
        else:
            preds = np.append(preds, logits.detach().cpu().numpy(), axis=0)
            out_label_ids = np.append(out_label_ids, inputs['labels'].detach().cpu().numpy(), axis=0)
    m = model.module if hasattr(model, 'module') else model
    if m.tagger_config.absa_type != 'crf': preds = np.argmax(preds, axis=-1)
    else:
        crf_logits = torch.cat(crf_logits, dim=0); crf_mask = torch.cat(crf_mask, dim=0)
        preds = m.tagger.viterbi_tags(logits=crf_logits, mask=crf_mask)
    result = compute_metrics_absa(preds, out_label_ids, evaluate_label_ids, TAGGING_SCHEMA)
    result['eval_loss'] = eval_loss / len(dataloader)
    return result

print('=== Dev set ===')
dev_result = evaluate(model, tokenizer, DATA_DIR, TASK_NAME, mode='dev')
print(dev_result)
print('\n=== Test set ===')
test_result = evaluate(model, tokenizer, DATA_DIR, TASK_NAME, mode='test')
print(test_result)

## 12. Inference - Thử dự đoán câu mới

In [ ]:
if TAGGING_SCHEMA == 'BIEOS':
    ABSA_ID2TAG = {0:'O',1:'EQ',2:'B-POS',3:'I-POS',4:'E-POS',5:'S-POS',
                   6:'B-NEG',7:'I-NEG',8:'E-NEG',9:'S-NEG',
                   10:'B-NEU',11:'I-NEU',12:'E-NEU',13:'S-NEU'}
elif TAGGING_SCHEMA == 'BIO':
    ABSA_ID2TAG = {0:'O',1:'EQ',2:'B-POS',3:'I-POS',4:'B-NEG',5:'I-NEG',6:'B-NEU',7:'I-NEU'}
else:
    ABSA_ID2TAG = {0:'O',1:'EQ',2:'T-POS',3:'T-NEG',4:'T-NEU'}

def predict_sentence(text, model, tokenizer):
    model.eval()
    words = text.strip().split()
    tokens, evaluate_ids = [], []
    for i, word in enumerate(words):
        evaluate_ids.append(len(tokens) + 1)
        tokens.extend(tokenizer.tokenize(word))
    tokens = [tokenizer.cls_token] + tokens + [tokenizer.sep_token]
    input_ids = torch.tensor([tokenizer.convert_tokens_to_ids(tokens)], dtype=torch.long).to(device)
    attention_mask = torch.ones_like(input_ids)
    token_type_ids = torch.zeros_like(input_ids)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = outputs[0]
    m = model.module if hasattr(model, 'module') else model
    if m.tagger_config.absa_type != 'crf':
        pred_ids = np.argmax(logits.detach().cpu().numpy(), axis=-1)[0]
    else:
        pred_ids = m.tagger.viterbi_tags(logits=logits, mask=attention_mask)[0]
    pred_tags = [ABSA_ID2TAG[pred_ids[i]] for i in evaluate_ids]
    if TAGGING_SCHEMA == 'OT': pred_tags = ot2bieos_ts(pred_tags)
    elif TAGGING_SCHEMA == 'BIO': pred_tags = ot2bieos_ts(bio2ot_ts(pred_tags))
    results = []
    for beg, end, sentiment in tag2ts(pred_tags):
        results.append((' '.join(words[beg:end+1]), sentiment))
    return results

test_sentences = [
    'The battery life is great but the screen is terrible',
    'Great laptop that offers many great features',
    'The food was amazing but the service was slow',
]
for sent in test_sentences:
    result = predict_sentence(text=sent, model=model, tokenizer=tokenizer)
    print(f'Input : {sent}')
    print(f'Output: {result}')
    print()

## 13. (Tùy chọn) Load checkpoint để evaluate lại

In [ ]:
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoint-{MAX_STEPS}'
if os.path.exists(CHECKPOINT_DIR):
    print(f'Loading checkpoint from {CHECKPOINT_DIR}')
    model_ckpt = BertABSATagger.from_pretrained(CHECKPOINT_DIR)
    model_ckpt.to(device)
    result = evaluate(model_ckpt, tokenizer, DATA_DIR, TASK_NAME, mode='test')
    print('Test result:', result)
else:
    print(f'Checkpoint not found: {CHECKPOINT_DIR}')